# 第2章 RAG 最小構成

外部文書から関連情報を検索し、その内容をプロンプトに混ぜて回答させる **RAG (Retrieval Augmented Generation)** の最小構成を作ります。

**所要時間**: 約30分

## 流れ

1. **Load**: 文書を読み込む
2. **Split**: チャンクに分割
3. **Embed & Store**: ベクトル化して Chroma に保存
4. **Retrieve**: 質問に近いチャンクを検索
5. **Generate**: 検索結果をプロンプトに入れて LLM が回答

## 0. 準備

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]
EMBED_MODEL_ID = os.environ["BEDROCK_EMBED_MODEL_ID"]

print("chat :", CHAT_MODEL_ID)
print("embed:", EMBED_MODEL_ID)

## 1. 文書のロードと分割

`TextLoader` でテキストファイルを読み、`RecursiveCharacterTextSplitter` でチャンクに分けます。

- `chunk_size`: 1チャンクあたりの最大文字数
- `chunk_overlap`: チャンク境界をまたいだ情報を取りこぼさないように、隣接チャンク間で重ねる文字数

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/sample.md", encoding="utf-8")
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)

print(f"分割後のチャンク数: {len(docs)}\n")
for i, d in enumerate(docs):
    print(f"--- chunk {i} ({len(d.page_content)} 字) ---")
    print(d.page_content)
    print()

## 2. 埋め込みとベクトルDB(Chroma)への保存

- 埋め込み: `BedrockEmbeddings`(Titan Text Embeddings V2)
- ベクトルDB: `Chroma`(`persist_directory` を指定するとローカルに永続化)

初回は埋め込み計算で少し時間がかかります。

> ⚠️ **再実行時の注意**: このセルを2回以上実行すると `./.chroma` に同じドキュメントが重複追加されます。
> 試しに何度も流す場合は、事前にターミナルで `rm -rf 02_rag/.chroma` を実行してから流してください。

In [ ]:
from langchain_aws import BedrockEmbeddings
from langchain_chroma import Chroma

embeddings = BedrockEmbeddings(
    model_id=EMBED_MODEL_ID,
    region_name=AWS_REGION,
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./.chroma",
)

print("件数:", len(vectorstore.get()["ids"]))

## 3. 類似検索の動作確認

質問に近いチャンクが返ることを確かめます。

In [ ]:
hits = vectorstore.similarity_search("LCELとは何ですか?", k=2)
for i, d in enumerate(hits):
    print(f"--- hit {i} ---")
    print(d.page_content)
    print()

## 4. RAGチェーンを組み立てる

LCELで以下を組みます:

```
{ context: retriever → 文字列整形,
  question: そのまま渡す }
        ↓
      prompt
        ↓
       llm
        ↓
    StrOutputParser
```

In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "あなたは社内ドキュメントに基づいて回答するアシスタントです。\n"
     "以下の参考情報のみを根拠に、簡潔に日本語で回答してください。\n"
     "参考情報に答えがない場合は『資料には記載がありません』と答えてください。\n\n"
     "### 参考情報\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 5. 質問してみる

In [ ]:
print(rag_chain.invoke("LCELとは何で、何が嬉しいですか?"))

In [ ]:
print(rag_chain.invoke("LangChainでBedrockを使うときのパッケージとクラス名は?"))

### 文書外の質問はどうなる?

プロンプトで「資料にない場合は記載がありませんと答える」よう指示しているので、ハルシネーション抑制が効くか確認します。

In [ ]:
print(rag_chain.invoke("今日の東京の天気は?"))

## まとめ

- Loader → Splitter → Embeddings → VectorStore でRAG用のインデックスが作れる
- `vectorstore.as_retriever()` で **検索器** に変換でき、LCELチェーンに組み込める
- 「参考情報」をプロンプトに注入する形にすれば、独自文書を根拠とした回答ができる

次は [03 エージェント](../03_agent/agent.ipynb)。LLMにツールを使わせる仕組みを体験します。